# 00 — Business Understanding

## Objetivo de negocio

Analizar las trazas OTDR/iOLM de los distintos puertos y RTUs del EXFO FMS para
identificar **patrones de atenuación de fibra** (ej. degradación que se acerca o
supera ~3dB respecto al baseline de la ruta, umbral que típicamente causa
pérdida del canal de comunicaciones) y así:

1. **Alertar de forma temprana** cuándo una ruta/puerto está cerca de una
   degradación que afecte el servicio.
2. **Priorizar** qué rutas requieren mantenimiento preventivo antes de que el
   servicio se vea afectado, en vez de reaccionar después de la falla.

## Criterios de éxito

- Un ranking de rutas por "riesgo" que un operador de NOC pueda usar para
  decidir dónde programar mantenimiento preventivo.
- Reducir el número de rutas que llegan a `FaultStatus = Detected` sin haber
  sido marcadas antes como en riesgo por el pipeline.
- Un pipeline reproducible (CRISP-DM) que se pueda re-correr periódicamente
  con datos nuevos del FMS.

## Sobre las etiquetas (weak supervision)

**No existe un histórico curado de incidentes reales confirmados** (ej. tickets
de NOC vinculados a fecha/ruta) que sirva como ground truth limpio. Se parte de
dos señales débiles disponibles en el propio FMS:

- La regla de dominio de atenuación (~3dB respecto al baseline de la ruta).
- El campo `FaultStatus` (`Detected`/`Cleared`) que el FMS ya calcula.

Estas señales se combinan en `src/weak_labels.py` como **weak labels**, no como
verdad de terreno. Todo el modelado y la evaluación posteriores (notebooks 03 y
04) deben interpretarse con esta limitación en mente: un modelo "perfecto"
contra el weak label solo demuestra que aprendió a replicar la regla, no que
predice fallas reales no vistas.

## Alcance de datos

Todas las rutas ópticas disponibles en el FMS (sin filtrar por sitio, RTU o
tipo de ruta).


## Prueba de conectividad

Confirma que el token de FMS está disponible antes de avanzar a los demás notebooks — evita que un notebook posterior se bloquee pidiendo un código TOTP interactivo.

In [ ]:
import sys
from pathlib import Path

SRC = Path("..").resolve() / "src"
sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

from fms_client import FmsClient

client = FmsClient()
print("[OK] Token obtenido correctamente.")


In [ ]:
# Prueba mínima: listar la primera página de rutas ópticas (no se interpreta
# el resultado todavía, eso es tarea del notebook 01).
sample = client.list_optical_routes(page_size=5, page=1)
print(type(sample))
